In [2]:
import pandas as pd

WHO_URL = "https://github.com/Priyankkoul/Life-Expectancy-WHO---Data-Analytics/blob/master/DATASET.csv?raw=true"

who = pd.read_csv(WHO_URL)

print("Shape:", who.shape)

# first_5_checks(who, "WHO")

Shape: (2938, 22)


In [13]:

import pandas as pd
from scipy import stats
import numpy as np

df = pd.read_csv("data/chess_games.csv")

In [14]:

rated = df[df['rated'] == True]['turns']
unrated = df[df['rated'] == False]['turns']
print(f"Rated:   n={len(rated)}, mean={rated.mean():.1f}")    # mean=62.3
print(f"Unrated: n={len(unrated)}, mean={unrated.mean():.1f}")  # mean=51.7
# Welch's t-test (no equal-variance assumption — safer default)
t_stat, p_value = stats.ttest_ind(rated, unrated, equal_var=False)
print(f"t={t_stat:.3f}, p={p_value:.6f}")   # t≈10.5, p≈0.000000
# p < 0.05 → reject H0 → rated games ARE significantly longer
# Effect size — Cohen's d
pooled_std = np.sqrt((rated.std()**2 + unrated.std()**2) / 2)
cohen_d = (rated.mean() - unrated.mean()) / pooled_std
print(f"Cohen's d: {cohen_d:.3f}")   # ≈ 0.31 — SMALL effect
# Real, but modest: ~10 extra turns on average
# Non-parametric alternative (our data isn't normal)
u, p_mw = stats.mannwhitneyu(rated, unrated, alternative='two-sided')
print(f"Mann-Whitney p={p_mw:.6f}")  # confirms same conclusion

Rated:   n=16155, mean=62.0
Unrated: n=3903, mean=54.3
t=13.279, p=0.000000
Cohen's d: 0.233
Mann-Whitney p=0.000000


In [17]:
df = pd.read_csv("data/chess_games.csv").copy()

# Create binary outcome (1 = white wins, 0 = otherwise)
df['white_wins'] = (df['winner'] == 'White').astype(int)

white_win_rate = df['white_wins'].mean()

print(f"White win rate: {white_win_rate:.3f}")  # ~0.495... actually close to 50%

White win rate: 0.499


In [18]:
from scipy.stats import t as t_dist
def confidence_interval(data, confidence=0.95):
    n = len(data); mean = data.mean(); se = stats.sem(data)
    h = se * t_dist.ppf((1 + confidence) / 2., n - 1)
    return mean - h, mean + h
rated_turns = df[df['rated'] == True]['turns']
lo, hi = confidence_interval(rated_turns)
print(f"95% CI: ({lo:.1f}, {hi:.1f})")   # (61.8, 62.8) — narrow, n=16182
# Bootstrap CI — no normality assumption required
def bootstrap_ci(data, n_boot=1000, confidence=0.95):
    rng = np.random.default_rng(42)
    means = [rng.choice(data, len(data), replace=True).mean() for _ in range(n_boot)]
    a = (1 - confidence) / 2
    return np.percentile(means, [a*100, (1-a)*100])
lo_b, hi_b = bootstrap_ci(rated_turns.values)
print(f"Bootstrap 95% CI: ({lo_b:.1f}, {hi_b:.1f})")  # very close to parametric

95% CI: (61.4, 62.5)
Bootstrap 95% CI: (61.4, 62.5)


In [21]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score
df['rating_diff'] = (
    df['white_rating']
    - df['black_rating']
)
X = df[['rating_diff']].values; y = df['turns'].values
for degree in [1, 2, 3, 5, 10]:
    poly = PolynomialFeatures(degree=degree)
    Xp = poly.fit_transform(X)
    model = LinearRegression()
    cv = cross_val_score(model, Xp, y, cv=5, scoring='neg_mean_squared_error')
    model.fit(Xp, y)
    train_rmse = np.sqrt(((y - model.predict(Xp))**2).mean())
    cv_rmse = np.sqrt(-cv.mean())
    print(f"degree={degree}  train={train_rmse:.2f}  cv={cv_rmse:.2f}")
# degree=1   train=34.07  cv=34.07   ← underfitting, gap≈0
# degree=10  train=33.98  cv=34.17   ← overfitting begins, gap widens

degree=1  train=33.55  cv=33.58
degree=2  train=33.37  cv=33.40
degree=3  train=33.37  cv=33.40
degree=5  train=33.32  cv=33.36
degree=10  train=33.54  cv=33.62


In [20]:
pip install scikit-learn

  Using cached scikit_learn-1.7.2-cp310-cp310-win_amd64.whl (8.9 MB)
  Using cached joblib-1.5.3-py3-none-any.whl (309 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl (18 kB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.2.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip
